In [2]:
import os
import torch

DIR1 = "/Users/shreyash/Documents/Sem3DL/AT2DL/Code"
DIR2 = "/Users/shreyash/Documents/Sem3DL/AT2DL/Code/SavedWeights"


# ✅ Robust loader (handles all save formats)
def load_state_dict(path):
    obj = torch.load(path, map_location='cpu')

    # Case 1: checkpoint dict
    if isinstance(obj, dict):
        if 'state_dict' in obj:
            return obj['state_dict']
        elif 'model_state_dict' in obj:
            return obj['model_state_dict']
        else:
            # filter only tensor entries
            return {k: v for k, v in obj.items() if torch.is_tensor(v)}

    # Case 2: full model
    else:
        return obj.state_dict()


# ✅ Core comparison logic
def compare_models(file1, file2):
    state1 = load_state_dict(file1)
    state2 = load_state_dict(file2)

    # Structure check
    if state1.keys() != state2.keys():
        return "❌ Different structure"

    exact_same = True
    approx_same = True

    for k in state1.keys():
        t1, t2 = state1[k], state2[k]

        # skip anything that's not tensor (extra safety)
        if not (torch.is_tensor(t1) and torch.is_tensor(t2)):
            continue

        if not torch.equal(t1, t2):
            exact_same = False

        if not torch.allclose(t1, t2, atol=1e-6):
            approx_same = False

    if exact_same:
        return "✅ EXACTLY SAME"
    elif approx_same:
        return "⚠️ Almost same (tiny float differences)"
    else:
        return "❌ DIFFERENT"


# ✅ Compare all matching files
files1 = set(f for f in os.listdir(DIR1) if f.endswith(".pth"))
files2 = set(f for f in os.listdir(DIR2) if f.endswith(".pth"))

common_files = sorted(files1 & files2)

print("\n🔍 Comparing models...\n")

for f in common_files:
    path1 = os.path.join(DIR1, f)
    path2 = os.path.join(DIR2, f)

    result = compare_models(path1, path2)
    print(f"{f}: {result}")


# ✅ Show missing files
print("\n📂 Missing Files:")
print("Only in DIR1:", files1 - files2)
print("Only in DIR2:", files2 - files1)


🔍 Comparing models...

googlenet_best.pth: ✅ EXACTLY SAME
mobilenetv3_best.pth: ✅ EXACTLY SAME
resnet50_best.pth: ✅ EXACTLY SAME

📂 Missing Files:
Only in DIR1: set()
Only in DIR2: set()
